**Voice-to-Grasp - analysis of continual-learning logs** (update counts, running accuracy, loss)
from `Continual_Learning_Experiments/<run>/log.csv`. Outputs were cleared for the release.

In [ ]:
# ==============================
# IMPORTS
# ==============================

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)

from scipy.stats import ttest_rel
from statsmodels.stats.contingency_tables import mcnemar

sns.set(style="whitegrid")

In [ ]:
# ==============================
# LOAD LOG DATA
# ==============================

log_path = r"D:\PythonEnvs\nlpglove\Continual_Learning_Experiments\Tester-9\log.csv"
df = pd.read_csv(log_path, encoding="latin1")


df.head()

In [ ]:
# Convert empty strings to NaN
df.replace("", np.nan, inplace=True)

# Add correctness column
df["is_correct"] = df["correct_label"].isna()

# Convert numeric columns
df["confidence"] = pd.to_numeric(df["confidence"], errors="coerce")
df["loss"] = pd.to_numeric(df["loss"], errors="coerce")
df["update_number"] = pd.to_numeric(df["update_number"], errors="coerce")
df["buffer_size"] = pd.to_numeric(df["buffer_size"], errors="coerce")

df.head()

In [ ]:
# ==============================
# OVERALL ACCURACY
# ==============================

overall_accuracy = df["is_correct"].mean()
print("Overall Accuracy:", round(overall_accuracy, 4))

In [ ]:
# ==============================
# PER-CLASS METRICS
# ==============================

true_labels = df["correct_label"].fillna(df["predicted_label"])
pred_labels = df["predicted_label"]

report = classification_report(true_labels, pred_labels)
print(report)

In [ ]:
# ==============================
# CONFUSION MATRIX
# ==============================

labels = sorted(true_labels.unique())

cm = confusion_matrix(true_labels, pred_labels, labels=labels)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels,
            yticklabels=labels)

plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# ==============================
# LEARNING CURVE
# ==============================

df["cumulative_accuracy"] = df["is_correct"].expanding().mean()

plt.figure(figsize=(8,5))
plt.plot(df["cumulative_accuracy"])
plt.title("Cumulative Accuracy Over Time")
plt.xlabel("Sample Index")
plt.ylabel("Accuracy")
plt.show()

In [ ]:
# ==============================
# LOSS CURVE
# ==============================

loss_df = df.dropna(subset=["loss"])

plt.figure(figsize=(8,5))
plt.plot(loss_df["update_number"], loss_df["loss"])
plt.title("Loss vs Update Number")
plt.xlabel("Update Number")
plt.ylabel("Loss")
plt.show()

In [ ]:
# ==============================
# UPDATE RATE
# ==============================

update_rate = df["update_number"].notna().sum() / len(df)
print("Update Rate:", round(update_rate, 4))

In [ ]:
# ==============================
# CONFIDENCE DISTRIBUTION
# ==============================

plt.figure(figsize=(8,5))
sns.histplot(df[df["is_correct"]]["confidence"], 
             label="Correct", color="green", kde=True)

sns.histplot(df[~df["is_correct"]]["confidence"], 
             label="Incorrect", color="red", kde=True)

plt.legend()
plt.title("Confidence Distribution")
plt.show()

In [ ]:
# ==============================
# CALIBRATION ANALYSIS
# ==============================

def compute_ece(confidences, correctness, n_bins=10):
    bins = np.linspace(0, 1, n_bins+1)
    ece = 0
    
    for i in range(n_bins):
        mask = (confidences >= bins[i]) & (confidences < bins[i+1])
        if np.sum(mask) > 0:
            bin_acc = np.mean(correctness[mask])
            bin_conf = np.mean(confidences[mask])
            ece += np.abs(bin_acc - bin_conf) * np.sum(mask) / len(confidences)
    
    return ece

ece = compute_ece(
    df["confidence"].values,
    df["is_correct"].astype(int).values
)

print("Expected Calibration Error:", round(ece, 4))

In [ ]:
# Example split
midpoint = len(df) // 2
df_kitchen = df.iloc[:midpoint]
df_repair = df.iloc[midpoint:]

kitchen_before = df_kitchen["is_correct"].mean()
kitchen_after = df_repair[df_repair.index < midpoint]["is_correct"].mean()

forgetting = kitchen_before - kitchen_after

print("Forgetting:", forgetting)

In [ ]:
# Plasticity: accuracy on last 20 samples
plasticity = df.tail(20)["is_correct"].mean()

# Stability: initial accuracy
stability = df.head(20)["is_correct"].mean()

print("Plasticity:", plasticity)
print("Initial Stability:", stability)

In [ ]:
# Samples that required update
hard_samples = df[~df["is_correct"]]

hard_samples.groupby("correct_label").size()

In [ ]:
# Example binary contingency
table = [[90, 10],
         [5, 95]]

result = mcnemar(table)
print("McNemar p-value:", result.pvalue)

In [ ]:
# Placeholder example
acc_initial = 0.82
acc_final = 0.87

bwt = acc_final - acc_initial
print("Backward Transfer:", bwt)

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(df["buffer_size"])
plt.title("Replay Buffer Growth")
plt.xlabel("Sample Index")
plt.ylabel("Buffer Size")
plt.show()

In [ ]:
from statsmodels.stats.proportion import proportion_confint

correct = df["is_correct"].sum()
total = len(df)

low, high = proportion_confint(correct, total, method='wilson')

print("95% CI:", (low, high))